In [1]:
import plotly.graph_objects as go
import diagnostics
import preprocess
import plots
import utils

c:\Users\joshu\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\joshu\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


### Setup

In [2]:
SNP_file = './data/S&P 500 Index.csv'
VIX_file = './data/S&P 500 VIX.csv'

In [3]:
# Scale is applied to log returns in order to improve numerical stability of models
SCALE = 100

### S&P500 GARCH Log Returns Forecasting

In [4]:
SNP_processed = preprocess.preprocess_data(SNP_file, 'price', 'log_returns')
SNP_processed = utils.upscale_columns(
  SNP_processed, ['log_returns'], SCALE
)
SNP_processed.head()

,price,log_returns
date,,
2015-01-05,2020.58,-1.844722
2015-01-06,2002.61,-0.893327
2015-01-07,2025.90,1.156272
2015-01-08,2062.14,1.773023
2015-01-09,2044.81,-0.843940


In [5]:
snp_garch = utils.rolling_garch_price_forecast(SNP_processed, 250, utils.Distribution.NORMAL)
snp_garch.head()

,price,log_returns,predicted_price,predicted_log_return,conditional_vol
date,,,,,
2015-01-05,2020.58,-1.844722,NaN,NaN,NaN
2015-01-06,2002.61,-0.893327,NaN,NaN,NaN
2015-01-07,2025.90,1.156272,NaN,NaN,NaN
2015-01-08,2062.14,1.773023,NaN,NaN,NaN
2015-01-09,2044.81,-0.843940,NaN,NaN,NaN


In [6]:
# Rescale back scaled columns
snp_garch = utils.downscale_columns(
  snp_garch, ['conditional_vol', 'log_returns', 'predicted_log_return'], SCALE
)
diagnostics.in_sample_diagnostics(snp_garch['predicted_log_return'], snp_garch['log_returns'], snp_garch['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.68657
Ljung-Box (residuals^2) p-value, 0.00341


In [7]:
snp_garch_var = snp_garch
snp_garch_var['VaR_99'] = snp_garch_var['predicted_log_return'].rolling(250).quantile(0.01)
diagnostics.compute_var_violations(snp_garch_var, 'VaR_99', 'predicted_log_return')

{'actual_exceedances': 30,
 'expected_exceedances': 20.150000000000016,
 'violation_ratio': 1.4888337468982618}

In [8]:
diagnostics.compute_rmse(snp_garch_var, 'log_returns', 'predicted_log_return')

0.011428109902533153

In [9]:
diagnostics.bernoulli_coverage_test(snp_garch_var, var_col='VaR_99', predicted_col='predicted_log_return')

(0.0397551860191947, 4.2283023343828745)

In [10]:
diagnostics.compute_hit_rate(predicted=snp_garch_var['predicted_log_return'], actual=snp_garch_var['log_returns'])


Hit Rate: 45.66%


0.45664280031821797

In [11]:
plots.plot_var_violations(snp_garch_var, var_col='VaR_99', predicted_col='predicted_log_return')

### S&P500 tGARCH Log Returns Forecasting

In [12]:
snp_tgarch = utils.rolling_garch_price_forecast(SNP_processed, 250, utils.Distribution.T)
snp_tgarch.head()

,price,log_returns,predicted_price,predicted_log_return,conditional_vol
date,,,,,
2015-01-05,2020.58,-1.844722,NaN,NaN,NaN
2015-01-06,2002.61,-0.893327,NaN,NaN,NaN
2015-01-07,2025.90,1.156272,NaN,NaN,NaN
2015-01-08,2062.14,1.773023,NaN,NaN,NaN
2015-01-09,2044.81,-0.843940,NaN,NaN,NaN


In [13]:
# Rescale back scaled columns
snp_tgarch = utils.downscale_columns(
  snp_tgarch, ['conditional_vol', 'log_returns', 'predicted_log_return'], SCALE
)
diagnostics.in_sample_diagnostics(snp_tgarch['predicted_log_return'], snp_tgarch['log_returns'], snp_tgarch['conditional_vol'])

Jarque-Bera test p-value: 0.00000
Ljung-Box (residuals) p-value, 0.81562
Ljung-Box (residuals^2) p-value, 0.26743


In [14]:
snp_tgarch_var = snp_tgarch
snp_tgarch_var['VaR_99'] = snp_tgarch_var['predicted_log_return'].rolling(250).quantile(0.01)
diagnostics.compute_var_violations(snp_tgarch_var, 'VaR_99', 'predicted_log_return')

{'actual_exceedances': 22,
 'expected_exceedances': 20.150000000000016,
 'violation_ratio': 1.0918114143920588}

In [15]:
diagnostics.compute_rmse(snp_tgarch_var, 'log_returns', 'predicted_log_return')

0.011419303994163249

In [16]:
diagnostics.bernoulli_coverage_test(snp_tgarch_var, var_col='VaR_99', predicted_col='predicted_log_return')

(0.6831554289614479, 0.16659545694133726)

In [17]:
diagnostics.compute_hit_rate(predicted=snp_tgarch_var['predicted_log_return'], actual=snp_tgarch_var['log_returns'])


Hit Rate: 43.00%


0.4299920445505171

In [18]:
plots.plot_var_violations(snp_tgarch_var, var_col='VaR_99', predicted_col='predicted_log_return')